# Models: Parametric vs DML

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import sys
from pathlib import Path
import statsmodels.api as sm
from econml.dml import LinearDML, CausalForestDML
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold


PROJECT_ROOT = Path.cwd().parent  # assumes notebooks/ is one level below root
sys.path.append(str(PROJECT_ROOT))

from src.load_data import load_feature

c:\Users\danil\anaconda3\envs\vair312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = load_feature()
df.head()

,host_response_rate,host_acceptance_rate,host_is_superhost,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,accommodates,bathrooms,...,amenity_Dedicated workspace,amenity_Toaster,amenity_Freezer,amenity_Shower gel,amenity_First aid kit,amenity_Dining table,amenity_Cleaning products,amenity_Self check-in,amenity_Fire extinguisher,amenity_Long term stays allowed
0,1.00,0.96,1,1.098612,1.791759,2,1,1,1,1.0,...,1,1,1,1,1,1,0,1,1,1
1,0.88,0.88,1,1.386294,2.833213,3,1,1,6,2.0,...,1,1,1,0,0,1,1,0,0,1
2,1.00,0.98,0,1.386294,4.691348,2,1,1,4,1.0,...,0,0,0,0,0,0,0,0,0,0
3,1.00,0.91,0,0.693147,0.693147,3,1,1,5,1.5,...,1,0,0,0,0,0,0,1,1,0
4,1.00,1.00,1,1.098612,1.609438,2,1,1,2,0.0,...,1,0,0,0,1,0,0,1,1,0


In [3]:
df.dtypes.value_counts()

bool       131
int64       46
float64     20
Name: count, dtype: int64

In [4]:
df.select_dtypes(include="object").columns.tolist()

[]

## Parametric model

In [5]:
#first spec without borough
borough_columns = [i for i in df.columns if "borough" in i ]

In [6]:
#outcome and treatment variable
y = df["log_price"]
d =df["log_rivals_500m"]

#exclude and controls
exclude = {"log_price", "log_rivals_500m", "loc_fe", "log_rivals_c_sq", "log_rivals_c"}
X = df[[c for c in df.columns if (c not in exclude)]]


#controls and treatment
Z = pd.concat([d,X], axis=1)

#adding constant
Z = sm.add_constant(Z, has_constant="add")

#changing data types
y = y.astype(float)
Z = Z.astype(float)

Z.shape



(42898, 194)

In [7]:
#running model
model = sm.OLS(y, Z).fit(
    cov_type="cluster",
    cov_kwds={"groups": df["loc_fe"]}
)

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:              log_price   R-squared:                       0.727
Model:                            OLS   Adj. R-squared:                  0.725
Method:                 Least Squares   F-statistic:                 3.547e+04
Date:                Thu, 26 Feb 2026   Prob (F-statistic):          3.35e-197
Time:                        15:44:30   Log-Likelihood:                -20674.
No. Observations:               42898   AIC:                         4.174e+04
Df Residuals:                   42704   BIC:                         4.342e+04
Df Model:                         193                                         
Covariance Type:              cluster                                         
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
const     

c:\Users\danil\anaconda3\envs\vair312\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 193, but rank is 94
  warnings.warn('covariance of constraints does not have full '


In [8]:
beta = model.params["log_rivals_500m"]
se   = model.bse["log_rivals_500m"]
ci_l, ci_u = model.conf_int().loc["log_rivals_500m"].tolist()

print(beta, se, ci_l, ci_u)

0.018666948934831038 0.0121524492257893 -0.005151413871667652 0.04248531174132973


### Non linear treatment

In [9]:
#outcome and treatment variable
y_nl = df["log_price"]
d_nl =df["log_rivals_c"]

#exclude and controls
exclude_nl = {"log_price", "log_rivals_500m", "loc_fe", "log_rivals_c"}
X_nl = df[[c for c in df.columns if (c not in exclude_nl) ]]


#controls and treatment
Z_nl = pd.concat([d_nl,X_nl], axis=1)

#adding constant
Z_nl = sm.add_constant(Z_nl, has_constant="add")

#changing data types
y_nl = y_nl.astype(float)
Z_nl = Z_nl.astype(float)

Z_nl.shape



(42898, 195)

In [10]:
#running model
model_nl = sm.OLS(y_nl, Z_nl).fit(
    cov_type="cluster",
    cov_kwds={"groups": df["loc_fe"]}
)

print(model_nl.summary())

                            OLS Regression Results                            
Dep. Variable:              log_price   R-squared:                       0.727
Model:                            OLS   Adj. R-squared:                  0.726
Method:                 Least Squares   F-statistic:                 4.152e+04
Date:                Thu, 26 Feb 2026   Prob (F-statistic):          1.18e-200
Time:                        15:44:30   Log-Likelihood:                -20664.
No. Observations:               42898   AIC:                         4.172e+04
Df Residuals:                   42703   BIC:                         4.341e+04
Df Model:                         194                                         
Covariance Type:              cluster                                         
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
const     

c:\Users\danil\anaconda3\envs\vair312\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 194, but rank is 95
  warnings.warn('covariance of constraints does not have full '


### Double Lasso PDS

In [11]:
#setting up controls, treatment adn outcome

y_lasso = "log_price"
d_lasso = "log_rivals_500m"
fe_lasso = "loc_fe"

#fe columns
fe_cols = [c for c in df.columns if c.startswith("fe_")]

exclude = {y_lasso, d_lasso, "rivals_500m", "price", "id",
            fe_lasso, "log_rivals_c_sq", "log_rivals_c"}

# Candidate controls for selection (exclude FE dummies from selection step)
X_cand_cols = [c for c in df.columns if c not in exclude and c not in fe_cols]

# FE dummies forced-in at the final stage
X_force_cols = fe_cols

### Double selection

In [12]:
#Variabels for lasso
X_cand = df[X_cand_cols].values
Y = df[y_lasso].values.astype(np.float64)
T = df[d_lasso].values.astype(np.float64)

# ElasticNetCV with l1_ratio = 1 for lasso
enet = ElasticNetCV(
    l1_ratio=1.0,
    alphas=None,
    cv=5,
    random_state=0,
    max_iter=10000
)

# Pipeline: standardize then fit
y_selector = Pipeline([("scaler", StandardScaler()), ("enet", enet)])
t_selector = Pipeline([("scaler", StandardScaler()), ("enet", enet)])

y_selector.fit(X_cand, Y)
t_selector.fit(X_cand, T)

coef_y = y_selector.named_steps["enet"].coef_
coef_t = t_selector.named_steps["enet"].coef_

S_y = set(np.array(X_cand_cols)[coef_y != 0])
S_t = set(np.array(X_cand_cols)[coef_t != 0])
S_union = sorted(list(S_y.union(S_t)))

print("Selected for Y:", len(S_y))
print("Selected for T:", len(S_t))
print("Union selected:", len(S_union))


Selected for Y: 90
Selected for T: 90
Union selected: 90


In [13]:
removed = [c for c in X_cand_cols if c not in S_union]
removed

['amenity_Washer', 'amenity_Cooking basics', 'amenity_Hot water kettle']

In [14]:
difference = [c for c in S_y if c not in S_t]
difference

[]

### PDS

In [15]:
Z_cols = [d_lasso] + S_union + X_force_cols
Z_lasso = df[Z_cols].copy()
Z_lasso = Z_lasso.astype(float)
Z_lasso = sm.add_constant(Z_lasso, has_constant="add")

model_pds = sm.OLS(df[y_lasso].values, Z_lasso).fit(
    cov_type="cluster",
    cov_kwds={"groups": df[fe_lasso]}
)

print(model_pds.summary())
print("\nPDS beta:", model_pds.params[d_lasso], "SE:", model_pds.bse[d_lasso])

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.727
Model:                            OLS   Adj. R-squared:                  0.725
Method:                 Least Squares   F-statistic:                 2.269e+04
Date:                Thu, 26 Feb 2026   Prob (F-statistic):          2.25e-187
Time:                        15:44:32   Log-Likelihood:                -20688.
No. Observations:               42898   AIC:                         4.176e+04
Df Residuals:                   42707   BIC:                         4.341e+04
Df Model:                         190                                         
Covariance Type:              cluster                                         
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
const     

c:\Users\danil\anaconda3\envs\vair312\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 190, but rank is 91
  warnings.warn('covariance of constraints does not have full '


### DML

In [24]:
X_dml = df[X_cand_cols + X_force_cols].values
X_dml = X_dml.astype(np.float64)

est = LinearDML(
    model_y=RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=20,
        n_jobs=-1,
        random_state=0
    ),
    model_t=RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=20,
        n_jobs=-1,
        random_state=0
    ),
    discrete_treatment=False,
    cv=KFold(n_splits=5, shuffle=True, random_state=0),
    random_state=0
)

est.fit(Y, T, X=X_dml)


In [25]:
ate = est.ate(X=X_dml)
print("Flexible DML ATE:", ate)


Flexible DML ATE: 0.00970857359298506


In [26]:
ci = est.ate_interval(X=X_dml)
print("CI:", ci)

CI: (np.float64(-0.0007030316084317423), np.float64(0.020120178794401863))


### Causal forest

In [ ]:
cf = CausalForestDML(
    model_y=RandomForestRegressor(n_estimators=200,
                                   min_samples_leaf=20, n_jobs=-1),
    model_t=RandomForestRegressor(n_estimators=200,
                                   min_samples_leaf=20, n_jobs=-1),
    n_estimators=1000,
    min_samples_leaf=50,
    random_state=42
)

cf.fit(Y, T, X=X_dml)